# ABP Steady-State Sanity Check

This notebook runs a longer WCA-ABP trajectory and checks whether simple observables have reached an approximately stationary regime.  For MIPS-like settings the density structure may continue to coarsen, so this notebook is a practical plateau check rather than a proof of stationarity.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath("../.."),
    os.path.abspath(".."),
    os.path.abspath("."),
    "/home/user1/CNEEP_v2",
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "ABP", "core.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from data.ABP import ABPParams, ContinuousABP, ABPFieldizer, recommended_center_grid_size

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Run a longer trajectory

In [ ]:
params = ABPParams(
    N=256,
    L=24.0,
    sigma=1.0,
    epsilon=1.0,
    mobility=1.0,
    force_clip=500.0,
    force_chunk_size=256,
    v0=12.0,
    Dr=1.0,
    Dt=0.01,
    dt=2.0e-4,
    seed=11,
    device=device,
)

grid_size = max(40, recommended_center_grid_size(params.L, params.sigma))
fieldizer = ABPFieldizer(params.L, grid_size, params.sigma, mode="center", include_orientation=False, clip_occupancy=False)
sim = ContinuousABP(params)

result = sim.simulate(
    B=3,
    burn_in=2_000,
    n_steps=6_000,
    save_interval=60,
    fieldizer=fieldizer,
    show_progress=True,
)

print("phi:", params.phi, "Pe:", params.Pe)
print("positions:", result["positions"].shape, "fields:", result["fields"].shape)

## 2. Build observables

In [ ]:
time = result["times"].numpy()
potential = result["potential"].numpy().mean(axis=1)
min_distance = result["min_distance"].numpy().mean(axis=1)
mean_force = result["mean_force_norm"].numpy().mean(axis=1)
fields = result["fields"][:, :, 0].numpy()  # [T, B, H, W]

def low_k_power(field_batch, kmax=3):
    vals = []
    for field in field_batch:
        f = field - field.mean()
        spec = np.abs(np.fft.rfft2(f)) ** 2
        low = spec[: kmax + 1, : kmax + 1].sum() - spec[0, 0]
        total = spec.sum() + 1e-12
        vals.append(low / total)
    return float(np.mean(vals))

low_power = np.array([low_k_power(fields[t]) for t in range(fields.shape[0])])

def running_mean(x, window=7):
    window = min(window, len(x))
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="same")

def plateau_report(name, x):
    q = max(2, len(x) // 4)
    middle = x[-2 * q : -q]
    late = x[-q:]
    scale = np.std(x[-2 * q :]) + 1e-12
    score = abs(late.mean() - middle.mean()) / scale
    print(f"{name:18s} middle={middle.mean():.5e} late={late.mean():.5e} score={score:.3f}")

plateau_report("WCA potential", potential)
plateau_report("min distance", min_distance)
plateau_report("low-k power", low_power)
plateau_report("mean |F|", mean_force)

## 3. Time-series plateau check

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
axes[0].plot(time, potential, alpha=0.45)
axes[0].plot(time, running_mean(potential), lw=2)
axes[0].set_ylabel("WCA U")

axes[1].plot(time, min_distance / params.sigma, alpha=0.45)
axes[1].plot(time, running_mean(min_distance / params.sigma), lw=2)
axes[1].axhline(1.0, color="k", linestyle="--", lw=1)
axes[1].set_ylabel("min r / sigma")

axes[2].plot(time, low_power, alpha=0.45)
axes[2].plot(time, running_mean(low_power), lw=2)
axes[2].set_ylabel("low-k power")

axes[3].plot(time, mean_force, alpha=0.45)
axes[3].plot(time, running_mean(mean_force), lw=2)
axes[3].set_ylabel("mean |F|")
axes[3].set_xlabel("time")

plt.tight_layout()
plt.show()

## 4. Early/middle/late density snapshots

In [ ]:
snap_ids = [0, len(time) // 2, len(time) - 1]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, t in zip(axes, snap_ids):
    img = fields[t, 0]
    im = ax.imshow(img.T, origin="lower", cmap="viridis")
    ax.set_title(f"t={time[t]:.3f}")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 5. Center-count audit over saved frames

In [ ]:
multi_center_counts = []
max_center_counts = []
for t in range(result["positions"].shape[0]):
    diag = fieldizer.diagnostics_dict(result["positions"][t].to(device))
    multi_center_counts.append(diag["multi_center_pixels"])
    max_center_counts.append(diag["max_center_count"])
multi_center_counts = np.asarray(multi_center_counts)
max_center_counts = np.asarray(max_center_counts)

plt.figure(figsize=(9, 3))
plt.plot(time, multi_center_counts, label="multi-center pixels")
plt.plot(time, max_center_counts, label="max center count")
plt.xlabel("time")
plt.ylabel("count statistic")
plt.title("Center-count audit")
plt.legend()
plt.tight_layout()
plt.show()

print("max multi-center pixels over saved frames:", int(multi_center_counts.max()))
print("max center count over saved frames:", int(max_center_counts.max()))
print("fieldizer dx:", fieldizer.dx)
print("hard-core diagnostic dx limit:", params.sigma / math.sqrt(2.0))